# Day 2 — 금융 데이터 분석 파이프라인

Upbit API로 수집한 암호화폐 데이터를 정제하고,  
투자 분석에 쓰이는 핵심 지표를 계산한 뒤 차트로 시각화합니다.

---

## 금융 데이터 기초 지식 (처음이라면 꼭 읽어두세요)

### 이번 시간에 계산할 지표 3가지

| 지표 | 컬럼명 | 의미 | 예시 |
|------|--------|------|------|
| 가격 변동 | `price_change` | 오늘 종가 - 오늘 시가 | +3,000원이면 하루 동안 3천원 올랐다 |
| 변동률(%) | `price_change_pct` | 시가 대비 변동 퍼센트 | +1.5%면 시가보다 1.5% 상승 마감 |
| 일중 변동폭 | `high_low_diff` | 고가 - 저가 | 하루 안에서 얼마나 크게 움직였는지 |
| 5일 이동평균 | `ma5` | 최근 5일 종가의 평균 | 단기 추세를 부드럽게 보여주는 선 |

---

### 캔들차트(OHLCV)와 각 가격의 관계

```
        고가(high) ← 하루 중 가장 높은 가격
           │
      ┌────┴────┐
      │  캔들   │  ← 시가(open)와 종가(close) 중 높은 쪽이 위
      └────┬────┘
           │
        저가(low) ← 하루 중 가장 낮은 가격

  시가(open)  : 하루 첫 거래 가격
  종가(close) : 하루 마지막 거래 가격 → 분석 기준!
```

> **왜 종가를 기준으로 분석할까요?**  
> 종가는 하루 동안의 모든 매수/매도가 반영된 **최종 결과값**이라  
> 가장 신뢰할 수 있는 가격으로 여겨집니다.

---

### 이동평균선(Moving Average, MA)이란?

**최근 N일간의 종가를 매일 평균 낸 값**을 선으로 이어 그린 것입니다.  
하루하루 가격의 출렁임을 부드럽게 만들어서 **추세**를 보여줍니다.

```
날짜       종가(close)    5일 이동평균(MA5)
--------------------------------------
1월 1일    100,000원       계산 불가 (데이터 부족)
1월 2일    102,000원       계산 불가
1월 3일     98,000원       계산 불가
1월 4일    105,000원       계산 불가
1월 5일    103,000원       101,600원 ← (100+102+98+105+103)÷5
1월 6일    107,000원       103,000원 ← (102+98+105+103+107)÷5
```

> 처음 4일은 5일치 데이터가 없어서 계산이 안 됩니다.  
> 이렇게 계산 불가한 빈칸을 **NaN(Not a Number)** 이라고 합니다.

---
## 0. 라이브러리 불러오기

파이썬의 기본 기능만으로는 데이터 분석이 힘들기 때문에  
전문가들이 미리 만들어둔 도구(라이브러리)를 가져와서 씁니다.

| 라이브러리 | 비유 | 우리가 쓰는 이유 |
|-----------|------|------------------|
| pyupbit | 배달앱 | 업비트 서버에서 코인 가격을 대신 받아다 줌 |
| pandas | 엑셀 | 데이터를 표(DataFrame) 형태로 쉽게 다룰 수 있게 해줌 |
| matplotlib | 그림판 | 선 그래프, 막대 그래프 등 차트를 그려줌 |
| matplotlib.dates | 달력 눈금 | 차트 x축에 날짜를 보기 좋게 표시해줌 |

In [ ]:
import pyupbit                           # 업비트 코인 가격 데이터 수집
import pandas as pd                      # 데이터를 표(DataFrame)로 다루기
import matplotlib.pyplot as plt          # 차트(그래프) 그리기
import matplotlib.dates as mdates        # 차트 x축에 날짜 포맷 적용
import warnings                          # 경고 메시지 제어

# 불필요한 경고 메시지 숨기기
# FutureWarning: 나중에 바뀔 수 있다는 경고인데, 학습용 코드에서는 일단 무시
warnings.filterwarnings('ignore')

print("라이브러리 불러오기 완료!")

---
## 1. 데이터 수집 및 통합

3종목의 데이터를 각각 받아온 뒤, **하나의 큰 표**로 합칩니다.  
이렇게 합쳐진 표에는 `ticker`(종목 코드) 컬럼이 추가되어  
어느 종목 데이터인지 구분할 수 있습니다.

```
KRW-BTC 데이터 (30행)  ┐
KRW-ETH 데이터 (30행)  ├→ 합치기(concat) → 하나의 DataFrame (90행)
KRW-SOL 데이터 (30행)  ┘
```

### Long Format(긴 형식) vs Wide Format(넓은 형식)

| 형식 | 설명 | 예시 |
|------|------|------|
| Wide Format | 종목별로 열(컬럼)을 따로 만들어 옆으로 넓어짐 | BTC_close, ETH_close, SOL_close ... |
| **Long Format** | 종목 구분을 `ticker` 열 하나로 표현, 행이 길어짐 | ticker=KRW-BTC, close=89250000 |  

> 우리는 **Long Format**을 씁니다.  
> 종목이 늘어나도 열을 추가할 필요 없이 행만 늘어나서 코드 수정이 쉽습니다.

In [ ]:
# ── 분석할 종목 리스트 ────────────────────────────────────────────
# 문제에서는 3종목을 쓰지만, 원하면 여기서 추가/변경 가능
TICKERS = ["KRW-BTC", "KRW-ETH", "KRW-SOL"]

# ── 데이터 수집 기간 ─────────────────────────────────────────────
# 5일 이동평균 계산 시, 처음 4일은 NaN이 됨
# 넉넉하게 60일치를 받아서 실제 분석은 최근 30일치로 자름
FETCH_DAYS = 60    # API에서 받아올 일수 (여유분 포함)
USE_DAYS   = 30    # 실제 사용할 일수 (최근 30일)

print(f"분석 종목: {TICKERS}")
print(f"API 수신: {FETCH_DAYS}일치, 실제 사용: {USE_DAYS}일치")

In [ ]:
def collect_ohlcv(tickers, fetch_days, use_days):
    """
    여러 종목의 OHLCV 데이터를 수집하고 하나의 DataFrame으로 합쳐서 반환합니다.

    매개변수(Parameter):
        tickers    : 수집할 종목 코드 리스트 (예: ["KRW-BTC", "KRW-ETH"])
        fetch_days : API에서 받아올 일수
        use_days   : 실제로 사용할 최근 일수 (fetch_days 이하여야 함)

    반환값(Return):
        성공한 종목들의 OHLCV 데이터를 세로로 쌓은 DataFrame
        컬럼: ticker, date, open, high, low, close, volume
    """
    # 각 종목 데이터를 임시로 담을 리스트
    # 나중에 pd.concat()으로 한번에 합칠 예정
    all_frames = []

    for ticker in tickers:
        print(f"  [{ticker}] 수집 중...")

        try:
            # pyupbit.get_ohlcv() : 업비트에서 일봉 데이터를 받아오는 함수
            # count=fetch_days    : 최근 n일치 요청
            # 반환되는 df의 인덱스(행 이름)가 날짜(datetime)로 설정되어 있음
            df = pyupbit.get_ohlcv(ticker, count=fetch_days)

            if df is None or df.empty:
                # 데이터가 비어있으면 건너뜀
                print(f"  [{ticker}] 수집 실패, 건너뜀")
                continue

            # ── 최근 use_days일치만 남기기 ─────────────────────
            # .tail(n) : DataFrame의 마지막 n행만 추출
            # 마지막 = 가장 최근 날짜 (날짜 오름차순 정렬이 기본)
            df = df.tail(use_days).copy()

            # ── 날짜 컬럼 추가 ──────────────────────────────────
            # 현재 날짜가 인덱스(index)에 있는데, 이를 'date' 컬럼으로 이동
            # .index    : DataFrame의 행 이름(인덱스) 접근
            # .date     : datetime에서 날짜 부분만 추출 (시간 제거)
            # pd.Series : 리스트처럼 생긴 1차원 데이터 구조
            df['date'] = pd.to_datetime(df.index.date)

            # ── 종목 코드(ticker) 컬럼 추가 ────────────────────
            # 나중에 합쳐졌을 때 어느 종목 데이터인지 구분하기 위해
            df['ticker'] = ticker

            # ── 인덱스를 0, 1, 2, ... 로 초기화 ────────────────
            # reset_index() : 현재 날짜 인덱스를 일반 열로 내리거나 버림
            # drop=True     : 날짜 인덱스를 열로 내리지 않고 그냥 버림
            #                 (이미 'date' 컬럼으로 따로 저장했기 때문)
            df = df.reset_index(drop=True)

            all_frames.append(df)
            print(f"  [{ticker}] 완료 ({len(df)}일치)")

        except Exception as e:
            # 예상치 못한 오류 발생 시 해당 종목을 건너뜀
            print(f"  [{ticker}] 오류 발생: {e}, 건너뜀")
            continue

    # ── 여러 종목 DataFrame을 세로로 합치기 ────────────────────────
    # pd.concat()  : 여러 DataFrame을 이어 붙이는 함수
    # axis=0       : 세로 방향으로 쌓기 (기본값, 행 수가 늘어남)
    # ignore_index=True : 합친 후 인덱스를 0, 1, 2... 로 새로 매기기
    if not all_frames:
        print("수집된 데이터가 없습니다!")
        return None

    combined_df = pd.concat(all_frames, axis=0, ignore_index=True)
    return combined_df

In [ ]:
# ── 데이터 수집 실행 ───────────────────────────────────────────────
print("=== 데이터 수집 시작 ===")
df = collect_ohlcv(TICKERS, FETCH_DAYS, USE_DAYS)
print(f"\n=== 수집 완료: 총 {len(df)}행 ===")

In [ ]:
# ── 수집된 데이터 확인 ─────────────────────────────────────────────

# .head() : 상위 5행 미리보기 (기본값 5, 괄호 안 숫자로 조절)
print("[수집된 데이터 상위 5행]")
print(df.head())

print()

# .info() : 컬럼명, 데이터 개수, 자료형(dtype) 한눈에 보기
# dtype 설명:
#   float64 → 소수점 있는 실수 (가격, 거래량 등)
#   object  → 문자열 (ticker 코드 등)
#   datetime64 → 날짜/시간 데이터
print("[컬럼 정보]")
df.info()

---
## 2. 데이터 전처리 — 필요한 컬럼만 추출

수집된 데이터에서 **분석에 필요한 컬럼만** 선택합니다.  
불필요한 열을 미리 제거하면 이후 작업이 깔끔해집니다.

### 사용할 컬럼

| 컬럼 | 설명 |
|------|------|
| `ticker` | 종목 코드 (KRW-BTC, KRW-ETH, KRW-SOL) |
| `date` | 날짜 |
| `open` | 시가 (하루 첫 거래 가격) |
| `high` | 고가 (하루 최고 가격) |
| `low` | 저가 (하루 최저 가격) |
| `close` | 종가 (하루 마지막 거래 가격) |
| `volume` | 거래량 (하루 동안 거래된 총 수량) |

In [ ]:
# ── 필요한 컬럼만 선택 ────────────────────────────────────────────
# DataFrame[ [컬럼1, 컬럼2, ...] ] : 원하는 컬럼들만 뽑아 새 DataFrame 생성
# 대괄호가 두 겹인 이유: 바깥 []은 DataFrame 접근, 안쪽 []은 컬럼 리스트
columns_needed = ['ticker', 'date', 'open', 'high', 'low', 'close', 'volume']
df = df[columns_needed].copy()

# ── 날짜 기준으로 정렬 ────────────────────────────────────────────
# sort_values() : 특정 컬럼 기준으로 행을 정렬
# by=['ticker', 'date'] : 먼저 종목 코드 순으로, 같은 종목 내에서는 날짜 순으로 정렬
# ascending=True        : 오름차순 (오래된 날짜 → 최신 날짜)
df = df.sort_values(by=['ticker', 'date'], ascending=True)

# 정렬 후 인덱스 초기화 (정렬하면 인덱스 순서가 뒤섞이므로 다시 0, 1, 2...)
df = df.reset_index(drop=True)

print("[전처리 후 데이터 미리보기]")
print(df.head(10))
print(f"\n총 {len(df)}행, {len(df.columns)}개 컬럼")

---
## 3. [필수 과제] 문제1: 종가와 시가의 차이 계산

### 이번 문제에서 추가할 컬럼 3가지

| 컬럼 | 계산식 | 의미 |
|------|--------|------|
| `price_change` | `close - open` | 하루 동안 가격이 얼마나 변했나 (원 단위) |
| `price_change_pct` | `price_change / open × 100` | 시가 대비 몇 % 변했나 |
| `high_low_diff` | `high - low` | 하루 안에서 최고가와 최저가의 차이 |

---

### 💡 각 지표를 왜 계산할까요?

**`price_change` (가격 변동)**
```
시가(open) = 90,000,000원, 종가(close) = 92,000,000원
→ price_change = +2,000,000원 (하루 동안 200만원 상승)

양수(+) : 종가 > 시가 → 상승 마감 (양봉)
음수(-) : 종가 < 시가 → 하락 마감 (음봉)
```

**`price_change_pct` (변동률)**
```
비트코인 +2,000,000원 vs 리플 +2,000,000원
→ 둘 다 같은 금액이 올랐지만, 가격 대비 상승률은 다름!
→ 퍼센트(%)로 비교해야 공정한 비교가 됨
```

**`high_low_diff` (일중 변동폭)**
```
변동폭이 크다 → 하루에 가격이 많이 움직였다 → 변동성이 크다 → 위험 but 기회
변동폭이 작다 → 가격이 안정적 → 변동성이 작다 → 안전 but 수익도 적음
```

In [ ]:
def add_price_features(df):
    """
    가격 관련 파생 컬럼 3개를 계산하여 DataFrame에 추가합니다.

    추가되는 컬럼:
        price_change     : 종가 - 시가 (원 단위 변동)
        price_change_pct : 시가 대비 변동률 (%)
        high_low_diff    : 고가 - 저가 (일중 변동폭)

    매개변수:
        df : open, high, low, close 컬럼이 있는 DataFrame

    반환값:
        파생 컬럼이 추가된 DataFrame
    """
    # .copy() : 원본 df를 건드리지 않도록 복사본에 작업
    # 원본을 직접 수정하면 나중에 실수로 잘못된 값을 다시 계산할 위험이 있음
    df = df.copy()

    # ── 컬럼1: price_change (가격 변동) ──────────────────────────
    # DataFrame에서 컬럼 간 사칙연산은 행(row)끼리 자동으로 맞춰서 계산됨
    # 즉, df['close'][0] - df['open'][0], df['close'][1] - df['open'][1], ...
    # 이런 계산을 한 줄로 표현 가능 (벡터 연산)
    df['price_change'] = df['close'] - df['open']

    # ── 컬럼2: price_change_pct (변동률, %) ──────────────────────
    # 계산식: (종가 - 시가) / 시가 × 100
    # → price_change를 이미 계산했으니 재사용
    # 예: price_change = 2,000,000 / open = 90,000,000
    #     → 2,000,000 / 90,000,000 × 100 ≈ 2.22%
    df['price_change_pct'] = (df['price_change'] / df['open']) * 100

    # ── 컬럼3: high_low_diff (일중 변동폭) ───────────────────────
    # 하루 동안 가격이 얼마나 큰 폭으로 움직였는지
    # 이 값이 클수록 당일 변동성이 높았음을 의미
    df['high_low_diff'] = df['high'] - df['low']

    return df

In [ ]:
# ── 함수 적용 ──────────────────────────────────────────────────────
df = add_price_features(df)

print("[문제1 결과 확인 — price_change, price_change_pct, high_low_diff 컬럼 추가됨]")
print()

# 새로 추가된 컬럼만 뽑아서 확인
# 종목(ticker), 날짜(date), 그리고 새 컬럼 3개를 함께 출력
check_cols = ['ticker', 'date', 'open', 'close', 'price_change', 'price_change_pct', 'high_low_diff']
print(df[check_cols].head(10).to_string(index=False))

# .to_string(index=False) : 인덱스 번호 없이 표처럼 보기 좋게 출력

In [ ]:
# ── 종목별 통계 요약 ───────────────────────────────────────────────
# 계산이 올바른지 종목별 평균을 보며 검증
print("[종목별 평균 price_change 및 변동률]")
print()

# groupby('ticker')  : 같은 ticker끼리 묶음
# [['price_change', 'price_change_pct']] : 확인할 컬럼 선택
# .mean()            : 묶인 그룹 내에서 평균 계산
# .round(2)          : 소수점 2자리까지 반올림
summary = df.groupby('ticker')[['price_change', 'price_change_pct', 'high_low_diff']].mean().round(2)
print(summary)

---
## 4. [필수 과제] 문제2: 5일 이동평균선 계산 및 결측치 처리

### 이동평균 계산 시 주의할 점 — NaN 문제

처음 4일은 과거 데이터가 5일치 없어서 이동평균을 계산할 수 없습니다.  
그 자리에 pandas는 자동으로 **NaN(빈값)** 을 채워 넣습니다.

```
날짜       close       MA5 계산 결과
----------------------------------------------
1월 1일    100,000    NaN  ← 과거 5일치 데이터 없음
1월 2일    102,000    NaN  ← 2일치만 있음
1월 3일     98,000    NaN  ← 3일치만 있음
1월 4일    105,000    NaN  ← 4일치만 있음
1월 5일    103,000  101,600  ← 5일치가 처음 모임! 계산 가능
```

### NaN 처리 방법: close 값으로 대체

NaN을 그냥 두면 차트에 **구멍**이 생기고, DB 저장 시 오류 날 수 있습니다.  
→ 이동평균이 없는 초기 구간은 **그날의 종가(close)** 로 대체합니다.

```
1월 1일:  MA5 = NaN  → close(100,000)으로 대체  → 100,000
1월 2일:  MA5 = NaN  → close(102,000)으로 대체  → 102,000
```

### 왜 0이나 평균값이 아닌 종가로 대체할까요?

> 종가는 그날의 실제 시장 가격이라  
> "이동평균이 없으면 일단 종가를 기준으로 본다"는 가장 자연스러운 처리 방식입니다.  
> 0으로 채우면 차트에서 갑자기 0원으로 뚝 떨어지는 현상이 생겨 분석이 왜곡됩니다.

### 종목별로 따로 계산해야 하는 이유

```
여러 종목이 하나의 DataFrame에 세로로 합쳐져 있는 상황:

날짜        ticker    close
2026-01-29  KRW-BTC  89,000,000   ← BTC 마지막 날
2026-01-30  KRW-BTC  90,000,000   ← BTC 마지막 날
2026-01-28  KRW-ETH   3,100,000   ← ETH 첫째 날  ← ⚠️ 종목이 바뀜!
2026-01-29  KRW-ETH   3,050,000   ← ETH 둘째 날

→ BTC 마지막 날과 ETH 첫째 날을 섞어서 평균 내면 의미 없는 숫자가 됨!
→ 반드시 ticker별로 나눠서 각각 이동평균을 계산해야 함
```

In [ ]:
def add_moving_average(df):
    """
    종목(ticker)별로 5일 이동평균(ma5)을 계산하고 NaN을 close로 대체합니다.

    처리 순서:
        1. ticker별로 데이터를 분리
        2. 각 그룹에서 close 기준 5일 이동평균 계산
        3. NaN을 해당 행의 close 값으로 대체
        4. 전체 DataFrame에 ma5 컬럼 추가

    매개변수:
        df : ticker, date, close 컬럼이 있는 DataFrame

    반환값:
        ma5 컬럼이 추가된 DataFrame
    """
    df = df.copy()

    # ── groupby로 ticker별 이동평균 계산 ──────────────────────────
    # groupby('ticker') : 같은 ticker끼리 묶음
    # ['close']         : 묶인 그룹 중 'close' 컬럼만 선택
    # .transform(lambda x: x.rolling(5).mean())
    #   → transform : 그룹별로 함수를 적용하되, 원래 DataFrame과 행 수를 동일하게 유지
    #   → lambda x  : x라는 이름으로 각 그룹(종목)의 close 시리즈를 받음
    #   → x.rolling(5).mean() : 그 그룹 내에서 5일 이동평균 계산
    # 결과: 각 행에 해당 종목의 5일 이동평균값이 들어옴
    df['ma5'] = df.groupby('ticker')['close'].transform(
        lambda x: x.rolling(5).mean()
    )

    # ── NaN을 close 값으로 대체 ───────────────────────────────────
    # fillna() : NaN(빈값)을 특정 값으로 채우는 함수
    # df['close'] : 같은 행의 close 값으로 채우겠다는 의미
    # 예) ma5 = NaN이고 close = 100,000이면 → ma5 = 100,000으로 대체
    df['ma5'] = df['ma5'].fillna(df['close'])

    return df

In [ ]:
# ── 함수 적용 ──────────────────────────────────────────────────────
df = add_moving_average(df)

print("[문제2 결과 확인 — ma5 컬럼 추가됨]")
print()

In [ ]:
# ── NaN이 남아있는지 검증 ──────────────────────────────────────────
# .isnull() : NaN인 셀을 True, 아닌 셀을 False로 표시
# .sum()    : True(=1)의 개수를 더함 → NaN 개수
nan_count = df['ma5'].isnull().sum()

if nan_count == 0:
    print("✅ NaN 없음: 결측치 처리 완료!")
else:
    print(f"⚠️  NaN이 {nan_count}개 남아있습니다. 처리 로직을 확인하세요.")

In [ ]:
# ── 종목별로 close와 ma5 비교 출력 ───────────────────────────────
print("[종목별 close vs ma5 비교 (앞 7일)]")
print()

for ticker in TICKERS:
    # 특정 ticker의 행만 필터링
    # df[조건] : 조건이 True인 행만 추출
    ticker_df = df[df['ticker'] == ticker][['date', 'close', 'ma5']].head(7)

    print(f"--- {ticker} ---")
    # .to_string(index=False) : 행 번호(인덱스) 없이 출력
    print(ticker_df.to_string(index=False))
    print()

---
## 5. 전처리 완료 — 컬럼 순서 정리

모든 컬럼이 추가된 최종 DataFrame의 컬럼 순서를 정돈합니다.  
순서를 명확히 하면 나중에 DB 저장이나 분석할 때 헷갈리지 않습니다.

```
[기존 순서 (뒤죽박죽)]                [정리된 순서]
ticker, date, open, high,      →    ticker, date,
low, close, volume,                 open, high, low, close, volume,
price_change, price_change_pct,     price_change, price_change_pct,
high_low_diff, ma5                  high_low_diff, ma5
```

In [ ]:
# ── 최종 컬럼 순서 정의 ────────────────────────────────────────────
# 리스트에 원하는 순서대로 컬럼 이름을 나열
final_columns = [
    'ticker',             # 종목 코드 (식별자)
    'date',               # 날짜
    'open',               # 시가 (원본 데이터)
    'high',               # 고가
    'low',                # 저가
    'close',              # 종가 (분석 기준)
    'volume',             # 거래량
    'price_change',       # 파생 지표 1: 가격 변동
    'price_change_pct',   # 파생 지표 2: 변동률(%)
    'high_low_diff',      # 파생 지표 3: 일중 변동폭
    'ma5',                # 파생 지표 4: 5일 이동평균
]

# DataFrame에 정의한 순서로 컬럼 재배치
df = df[final_columns]

print("[최종 DataFrame 컬럼 순서]")
print(list(df.columns))

print()
print("[최종 DataFrame 미리보기 — 상위 5행]")
print(df.head())

In [ ]:
# ── 최종 통계 요약 ─────────────────────────────────────────────────
# .describe() : 숫자형 컬럼의 기초 통계량 (개수, 평균, 표준편차, 최솟값, 최댓값 등)
print("[종목별 price_change_pct 기초 통계]")
print()
print(df.groupby('ticker')['price_change_pct'].describe().round(2))

---
## 6. [도전 과제] 문제3: 종가와 5일 이동평균선 시각화

### matplotlib과 plotly의 차이점

| | matplotlib | plotly (1주차) |
|--|-----------|----------------|
| 인터랙션 | ❌ 정적 이미지 | ✅ 마우스 조작 가능 |
| 파일 저장 | ✅ 이미지(.png) 저장 쉬움 | PDF/PNG 별도 설정 필요 |
| 사용 난이도 | 상대적으로 단순 | 다양한 옵션 있음 |
| 속도 | 빠름 | 약간 느림 |

> 이번에는 matplotlib을 써서 가볍고 깔끔한 정적 차트를 만듭니다.  
> 보고서에 넣을 이미지를 뽑을 때 matplotlib이 유용합니다.

---

### 차트 레이아웃 구조

```
┌──────────────────────────────────────────────────────────┐
│              암호화폐 종가 및 5일 이동평균선               │
├─────────────────┬──────────────────┬──────────────────────┤
│   KRW-BTC       │   KRW-ETH        │   KRW-SOL            │
│                 │                  │                       │
│  ▔▔▔▔▔▔▔━━━    │  ▁▂▃▂▁▂▃▄▃▂▁   │  ▁▂▃▄▅▄▃▄▅▆▅▄▃▂▁     │
│                 │                  │                       │
│ ─ 종가          │ ─ 종가           │ ─ 종가                │
│ - MA5           │ - MA5            │ - MA5                 │
├─────────────────┴──────────────────┴──────────────────────┤
│    01-01   01-08   01-15   01-22   01-29                   │
└──────────────────────────────────────────────────────────┘
```

### 서브플롯(subplot)이란?

> 하나의 figure(전체 그림) 안에 여러 개의 axes(개별 차트)를 배치하는 것입니다.  
> `plt.subplots(1, 3)` → 1행 3열 = 3칸짜리 차트를 만듭니다.

In [ ]:
def prepare_plot_data(df):
    """
    시각화 전 데이터 준비 작업을 수행합니다.
    date 컬럼을 datetime 타입으로 변환하고, 종목별로 데이터를 분리합니다.

    매개변수:
        df : ticker, date, close, ma5 컬럼이 있는 DataFrame

    반환값:
        ticker별 DataFrame을 담은 딕셔너리
        예: {"KRW-BTC": DataFrame, "KRW-ETH": DataFrame, ...}
    """
    df = df.copy()

    # ── date 컬럼을 datetime 타입으로 변환 ─────────────────────────
    # pd.to_datetime() : 문자열, 날짜 등을 pandas datetime64 타입으로 변환
    # matplotlib의 mdates(날짜 포맷)는 datetime 타입을 필요로 함
    # 이미 datetime이어도 다시 변환해도 무방함 (중복 변환 안전)
    df['date'] = pd.to_datetime(df['date'])

    # ── 종목별로 데이터 분리 ──────────────────────────────────────
    # 딕셔너리에 {ticker명: 해당 ticker DataFrame} 형태로 저장
    ticker_data = {}
    for ticker in df['ticker'].unique():
        # .unique() : 중복 없이 유일한 값들만 반환 (종목 코드 목록)

        # 해당 ticker 행만 필터링 후 날짜 기준 정렬
        ticker_df = df[df['ticker'] == ticker].sort_values('date').copy()
        ticker_data[ticker] = ticker_df

    return ticker_data

In [ ]:
def plot_close_and_ma5(ticker_data):
    """
    3종목의 종가(close)와 5일 이동평균(ma5)을 서브플롯으로 시각화합니다.

    차트 구성:
        - 1행 3열 서브플롯 (종목당 1칸)
        - 검정 실선: 종가(close)
        - 빨간 점선: 5일 이동평균(ma5)
        - x축: 날짜 (MM-DD 형식)
        - y축: 가격 (원 단위)

    매개변수:
        ticker_data : prepare_plot_data()에서 반환된 딕셔너리
    """
    # ── figure(전체 그림) 생성 ────────────────────────────────────
    # plt.subplots(행 수, 열 수) : 서브플롯 틀 생성
    # nrows=1, ncols=3          : 1행 3열 = 3칸
    # figsize=(너비, 높이)       : 전체 그림 크기 (인치 단위, 숫자 크게 할수록 넓어짐)
    # fig  : 전체 그림 객체 (배경)
    # axes : 각 서브플롯(차트 칸) 객체들의 배열 [ax0, ax1, ax2]
    fig, axes = plt.subplots(nrows=1, ncols=len(ticker_data),
                              figsize=(18, 5))

    # ── 각 종목별로 차트 그리기 ────────────────────────────────────
    # enumerate() : 인덱스(i)와 값(ticker)을 동시에 반환
    # i=0 → 첫 번째 칸(axes[0]), i=1 → 두 번째 칸(axes[1]) ...
    for i, ticker in enumerate(ticker_data):
        ax  = axes[i]           # i번째 서브플롯 선택
        tdf = ticker_data[ticker]  # 해당 ticker 데이터

        # ── 종가 선 그리기 ────────────────────────────────────────
        # ax.plot(x축 데이터, y축 데이터, 옵션)
        # color     : 선 색상
        # linewidth : 선 굵기 (숫자 클수록 굵음)
        # label     : 범례에 표시될 이름
        ax.plot(tdf['date'], tdf['close'],
                color='black',
                linewidth=1.5,
                label='종가(close)')

        # ── 5일 이동평균선 그리기 ─────────────────────────────────
        # linestyle='--' : 점선 (실선은 '-', 점선은 '--', 점점선은 '-.')
        # alpha         : 투명도 (0=완전투명, 1=불투명, 0.8=약간 투명)
        ax.plot(tdf['date'], tdf['ma5'],
                color='red',
                linewidth=1.2,
                linestyle='--',
                alpha=0.8,
                label='MA5 (5일 이동평균)')

        # ── 차트 제목 ──────────────────────────────────────────────
        # fontsize : 글자 크기 (pt 단위)
        ax.set_title(ticker, fontsize=13, fontweight='bold')

        # ── x축 라벨 ──────────────────────────────────────────────
        ax.set_xlabel('날짜', fontsize=10)

        # ── y축 라벨 (첫 번째 칸에만 표시) ───────────────────────
        # 모든 칸에 표시하면 중복되어 지저분하므로, 첫 칸에만 표시
        if i == 0:
            ax.set_ylabel('가격 (원)', fontsize=10)

        # ── x축 날짜 포맷 설정 ────────────────────────────────────
        # mdates.DateFormatter('%m-%d') : 날짜를 '01-05' 형식으로 표시
        # %Y = 4자리 연도, %m = 2자리 월, %d = 2자리 일
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))

        # 날짜 눈금을 약 5개로 자동 조절 (너무 빽빽하지 않게)
        # MaxNLocator(5) : 최대 5개의 눈금만 표시
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=6))

        # x축 날짜 텍스트를 45도 기울여서 겹치지 않게
        # ha='right' : 텍스트 오른쪽 정렬 (기울였을 때 위치 보정)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)

        # ── y축 숫자 포맷 ─────────────────────────────────────────
        # 89250000 같은 큰 숫자에 쉼표 삽입: 89,250,000
        # FuncFormatter : 사용자가 직접 포맷 함수를 정의
        # lambda x, _ : 실제 값(x)을 받아 포맷된 문자열 반환
        # f'{x:,.0f}' : 쉼표 + 소수점 없이
        import matplotlib.ticker as mticker
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f'{x:,.0f}')
        )

        # ── 범례 표시 ──────────────────────────────────────────────
        # loc='best' : matplotlib이 범례가 차트 요소와 겹치지 않는 위치를 자동 선택
        ax.legend(loc='best', fontsize=9)

        # ── 배경 격자 추가 ─────────────────────────────────────────
        # grid() : 격자 줄 표시
        # alpha=0.3 : 격자를 연하게 (차트 선이 잘 보이도록)
        # linestyle='--' : 격자도 점선으로
        ax.grid(True, alpha=0.3, linestyle='--')

    # ── 전체 제목 ──────────────────────────────────────────────────
    # suptitle() : 전체 figure(그림)의 제목 (각 서브플롯 제목과 별개)
    # y=1.02     : 제목 위치를 그림 위로 살짝 올림 (겹침 방지)
    fig.suptitle('암호화폐 종가 및 5일 이동평균선 (최근 30일)',
                 fontsize=15, fontweight='bold', y=1.02)

    # ── 여백 자동 조절 ────────────────────────────────────────────
    # tight_layout() : 서브플롯 간 여백을 자동으로 최적화
    # 서브플롯 제목, x축 날짜 등이 겹치지 않게 자동 정렬
    plt.tight_layout()

    # ── 차트 출력 ──────────────────────────────────────────────────
    plt.show()

    print("차트 출력 완료!")

In [ ]:
# ── 시각화 실행 ───────────────────────────────────────────────────

# 1단계: 시각화용 데이터 준비 (날짜 타입 변환 + 종목별 분리)
ticker_data = prepare_plot_data(df)

# 2단계: 차트 출력
plot_close_and_ma5(ticker_data)

---
## Day 2 완료 체크리스트

- [ ] 3종목 데이터 수집 완료 (30일치)
- [ ] 필요한 컬럼만 추출 및 정렬 완료
- [ ] **[필수] 문제1**: `price_change`, `price_change_pct`, `high_low_diff` 컬럼 추가 완료
- [ ] **[필수] 문제2**: `ma5` 컬럼 추가 완료, NaN 결측치 없음 확인
- [ ] 컬럼 순서 최종 정리 완료
- [ ] **[도전] 문제3**: 종가 + MA5 서브플롯 차트 정상 출력

---

## Day 3 예고

오늘 완성한 `df` (가격 지표가 추가된 DataFrame)를  
**SQLite 데이터베이스에 저장**하고, SQL로 조회하는 방법을 배웁니다.  
데이터를 메모리(RAM)가 아닌 파일로 영구 저장하는 방법입니다!